# Stage D v2 (bi-encoder + LLM disambiguation) – Google Colab

Run **Stage D v2** (entity linking with SapBERT bi-encoder + **LLM disambiguation**, NSSC/BioLinker-style) in Colab with GPU.

**Setup:** Runtime → Change runtime type → **T4 GPU**. Then run cells in order.

**You need:**
1. Project ZIP (e.g. from `create_colab_zip.py`) or project folder with `pipeline/`, `config.py`
2. `stage_c_statements_with_entities.json` (from `outputs/STAGE_C_v1/`)
3. `UMLS.csv` (from `input/` – or use a subset for faster index build)
4. **Hugging Face token** and LLaMA 3.2 license accepted (for Step 5a, like NSSC/BioLinker)

**Output:** `stage_d_candidate_statements.json` in `outputs/STAGE_D_v2/` (download at the end).

## Step 1: Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu torch transformers accelerate
print("[OK] Dependencies installed!")

## Step 2: Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable T4 GPU: Runtime → Change runtime type"
print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Upload project and data

In [ ]:
from google.colab import files
import zipfile
import os
import sys
from pathlib import Path

print("[*] Upload your project ZIP (or skip and upload files manually)...")
uploaded = files.upload()
project_dir = "/content"
for fn in uploaded:
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, "r") as z:
            z.extractall("/content")
        break
for d in os.listdir("/content"):
    p = Path("/content") / d
    if p.is_dir() and (p / "pipeline").exists():
        project_dir = str(p)
        break
if (Path(project_dir) / "pipeline").exists():
    sys.path.insert(0, project_dir)
    os.chdir(project_dir)
    print(f"[OK] Project: {project_dir}")
else:
    print("[!] No pipeline/ found. Upload a ZIP that contains the Thesis project.")
    project_dir = "/content"

In [ ]:
from google.colab import files
from pathlib import Path

STAGE_C_PATH = Path("/content/stage_c_statements_with_entities.json")
UMLS_PATH = Path(project_dir) / "input" / "UMLS.csv"

print("[*] Upload stage_c_statements_with_entities.json (from outputs/STAGE_C_v1/)...")
up = files.upload()
for f in up:
    if "stage_c" in f and f.endswith(".json"):
        with open(STAGE_C_PATH, "wb") as out:
            out.write(up[f] if isinstance(up[f], bytes) else open(f, "rb").read())
        print(f"[OK] Saved to {STAGE_C_PATH}")
        break

if not UMLS_PATH.exists():
    print("[*] Upload UMLS.csv (from input/)...")
    up2 = files.upload()
    for f in up2:
        if "UMLS" in f and f.endswith(".csv"):
            UMLS_PATH.parent.mkdir(parents=True, exist_ok=True)
            with open(UMLS_PATH, "wb") as out:
                out.write(up2[f] if isinstance(up2[f], bytes) else open(f, "rb").read())
            print(f"[OK] Saved to {UMLS_PATH}")
            break
else:
    print(f"[OK] Using UMLS at {UMLS_PATH}")

## Step 4: Build bi-encoder index from UMLS

Runs on GPU. Use `max_concepts` to limit size (e.g. 200000) for a quicker run; set to `None` for full UMLS.

In [ ]:
import sys
sys.path.insert(0, project_dir)

from pathlib import Path
from pipeline.models.entities_linker import build_biencoder_index

INDEX_DIR = Path("/content/stage_d_v2_index")
MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
MAX_CONCEPTS = 200000

if not UMLS_PATH.exists():
    raise FileNotFoundError(f"UMLS not found: {UMLS_PATH}. Upload it in Step 3.")

print("[*] Building bi-encoder index (this may take 10–30 min for 200k concepts)...")
n, _ = build_biencoder_index(
    str(UMLS_PATH),
    MODEL_NAME,
    str(INDEX_DIR),
    batch_size=50000,
    max_concepts=MAX_CONCEPTS,
)
print(f"[OK] Indexed {n} concepts in {INDEX_DIR}")

## Step 5a: Load LLaMA from Hugging Face for disambiguation (NSSC/BioLinker-style)

**Required for NSSC/BioLinker-style linking.** Both papers use an LLM to disambiguate among candidates; this step loads LLaMA from Hugging Face and builds the disambiguator. You need: (1) a [Hugging Face token](https://huggingface.co/settings/tokens), (2) accept the [LLaMA 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct). In Colab: add token as secret `HF_TOKEN` or paste when prompted. Set `USE_LLM_DISAMBIGUATION = False` only if you cannot run LLaMA (e.g. GPU memory); then the linker falls back to type reranking only (weaker).

In [ ]:
USE_LLM_DISAMBIGUATION = True

DISAMBIGUATOR = None
if USE_LLM_DISAMBIGUATION:
    import sys
    sys.path.insert(0, project_dir)
    # Log in to Hugging Face (required for gated LLaMA)
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
    except Exception:
        _hf_token = input("Paste your Hugging Face token (from https://huggingface.co/settings/tokens): ").strip()
    if _hf_token:
        import huggingface_hub
        huggingface_hub.login(token=_hf_token)
        print("[OK] Logged in to Hugging Face")
    from pipeline.models import ValidationModel, make_llm_disambiguator
    print("[*] Loading LLaMA from Hugging Face (this may take a few minutes)...")
    LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
    _vm = ValidationModel(model_name=LLAMA_MODEL)
    DISAMBIGUATOR = make_llm_disambiguator(_vm.run_inference, max_new_tokens=50)
    print(f"[OK] Disambiguator ready (model: {LLAMA_MODEL})")
else:
    print("[*] LLaMA skipped (USE_LLM_DISAMBIGUATION=False). Linker will use type reranking only, not NSSC/BioLinker-style.")

## Step 5: Run Stage D v2 (link entities)

In [ ]:
import json
import sys
import importlib
sys.path.insert(0, project_dir)

# Reload so edits to entities_linker.py are picked up without restarting runtime
import pipeline.models.entities_linker
importlib.reload(pipeline.models.entities_linker)

from pathlib import Path
from pipeline.data import StatementsWithMedicalEntities, CandidateStatements
from pipeline.models.entities_linker import BiEncoderLinker

if not STAGE_C_PATH.exists():
    raise FileNotFoundError("Stage C file not found. Upload it in Step 3.")

print("[*] Loading Stage C output...")
with open(STAGE_C_PATH, "r", encoding="utf-8") as f:
    stage_c_data = json.load(f)

statements_with_entities = StatementsWithMedicalEntities()
for stmt in stage_c_data["statements"]:
    statements_with_entities.add_statement(stmt)
table_triples_raw = stage_c_data.get("table_triples", [])
print(f"    Statements: {statements_with_entities.count()}, Table triples: {len(table_triples_raw)}")

try:
    _use_llm = USE_LLM_DISAMBIGUATION
    _disambiguator = DISAMBIGUATOR
except NameError:
    _use_llm = False
    _disambiguator = None

print("[*] Loading bi-encoder linker...")
linker = BiEncoderLinker(
    model_name=MODEL_NAME,
    index_dir=str(INDEX_DIR),
    top_k=16,
    min_link_score=0.72,
    use_type_rerank=True,
    prefer_shorter_concept=True,
    disambiguator=_disambiguator if _use_llm else None,
)
print(f"    {linker.get_umls_stats()}")
if _use_llm and _disambiguator:
    print("    LLaMA disambiguation: ON")

print("[*] Linking entities in statements...")
result = CandidateStatements()
for i, stmt in enumerate(statements_with_entities.get_all()):
    text = stmt.get("text", "")
    raw_entities = stmt.get("entities", [])
    linked = linker.link_entities(entities=raw_entities, context_text=text)
    result.add_statement({
        "chunk_id": stmt.get("chunk_id"),
        "page": stmt.get("page"),
        "source": stmt.get("source", ""),
        "text": text,
        "original_text": stmt.get("original_text", text),
        "entities": linked,
    })
    if (i + 1) % 200 == 0:
        print(f"    Processed {i + 1}/{statements_with_entities.count()} statements")

print("[*] Linking entities in table triples...")
table_triples_enriched = []
for idx, triple in enumerate(table_triples_raw):
    out = dict(triple)
    entities = triple.get("entities", [])
    if entities:
        triple_context = " ".join(
            part.strip() for part in (
                triple.get("subject", ""),
                triple.get("predicate", ""),
                triple.get("object", ""),
            ) if isinstance(part, str) and part.strip()
        )
        linker_input = [{"text": e.get("text", ""), "label": e.get("label", "")} for e in entities]
        for i, e in enumerate(entities):
            if i < len(linker_input) and "score" in e:
                linker_input[i]["score"] = e["score"]
        linked = linker.link_entities(linker_input, context_text=triple_context)
        out["entities"] = linked
    table_triples_enriched.append(out)

out_dir = Path(project_dir) / "outputs" / "STAGE_D_v2"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "stage_d_candidate_statements.json"
output_data = {
    "metadata": {
        "stage": "d",
        "stage_d_version": "v2",
        "description": "Candidate statements and table triples with UMLS-linked entities (bi-encoder)",
        "total_statements": result.count(),
        "total_candidates": result.count_candidates(),
        "total_table_triples": len(table_triples_enriched),
        "umls_linking": True,
    },
    "statements": result.get_all(),
    "table_triples": table_triples_enriched,
}
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)
print(f"[OK] Wrote {out_file}")

## Step 6: Download result

In [ ]:
from google.colab import files
from pathlib import Path

out_file = Path(project_dir) / "outputs" / "STAGE_D_v2" / "stage_d_candidate_statements.json"
if out_file.exists():
    files.download(str(out_file))
    print("[OK] Downloaded stage_d_candidate_statements.json")
else:
    print("[!] Run Step 5 first.")